# LC 300 — Longest Increasing Subsequence
**Day 54 | Mixed Review Sprint | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Maintain a <code>tails</code> array
where <code>tails[i]</code> is the smallest tail of all
increasing subsequences of length <code>i+1</code>. For
each number, use <code>bisect_left</code> to find its
position: replace or append. Length of <code>tails</code>
is the LIS length. O(n log n).
</div>

## Official Problem Statement

Given an integer array `nums`, return the length of the
longest **strictly increasing subsequence**.

A **subsequence** is a sequence derived from the array
by deleting some or no elements without changing the
order of the remaining elements.

**Constraints:**
- `1 <= nums.length <= 2500`
- `-10^4 <= nums[i] <= 10^4`

**Follow-up:** Can you come up with an algorithm that
runs in O(n log n) time complexity?

## What This Is Actually Asking

Find the longest chain of numbers from the array where
each number is strictly greater than the previous — but
elements don't need to be adjacent. The O(n²) DP is
straightforward: `dp[i]` = LIS ending at index `i`.
The O(n log n) "patience sort" insight is cleverer:
keep the smallest possible tail for each length, so
future numbers have the best chance of extending it.
Binary search finds exactly where each number fits.

## Walk Through an Example by Hand

```
nums = [10, 9, 2, 5, 3, 7, 101, 18]

tails = []   (empty at start)

num=10: tails=[]   bisect_left=0   append  tails=[10]
num=9:  tails=[10] bisect_left=0   replace tails=[9]
num=2:  tails=[9]  bisect_left=0   replace tails=[2]
num=5:  tails=[2]  bisect_left=1   append  tails=[2,5]
num=3:  tails=[2,5] bisect_left=1  replace tails=[2,3]
num=7:  tails=[2,3] bisect_left=2  append  tails=[2,3,7]
num=101:tails=[2,3,7] bisect_left=3 append tails=[2,3,7,101]
num=18: tails=[2,3,7,101] bisect_left=3 replace tails=[2,3,7,18]

len(tails) = 4   Answer: 4
(actual LIS: [2, 3, 7, 101] or [2, 3, 7, 18])
```

## The Picture

```
tails array after each step for [10,9,2,5,3,7,101,18]:

  num   tails            action
  ---   -----            ------
   10   [10]             append
    9   [9]              replace at pos 0 (9 < 10)
    2   [2]              replace at pos 0 (2 < 9)
    5   [2, 5]           append  (5 > 2, pos 1)
    3   [2, 3]           replace at pos 1 (3 < 5)
    7   [2, 3, 7]        append  (7 > 3, pos 2)
  101   [2, 3, 7, 101]   append  (101 > 7, pos 3)
   18   [2, 3, 7, 18]    replace at pos 3 (18 < 101)

  tails is always SORTED (invariant).
  tails[i] = best (smallest) tail for a length-(i+1) IS.

  bisect_left(tails, num):
    -> finds first position where tails[pos] >= num
    -> if pos == len(tails): append (new longest)
    -> else: replace (better tail for same length)
```

## When To Use This Pattern

- When asked for the **length of a longest / shortest
  subsequence** with an ordering constraint, think
  **patience sort + binary search**.
- When elements don't need to be adjacent but order
  must be preserved, think **subsequence DP**.
- When O(n²) DP is too slow and you need O(n log n),
  think **sorted tails array + bisect**.
- When you need to keep the "best so far" option for
  future elements, think **greedy replacement**.
- LIS is a building block for problems like Box
  Stacking, Russian Doll Envelopes, and patience
  sorting card game variants.

## The Approach

Maintain a list `tails` (always sorted). For each
number, use `bisect_left` to find the insertion point.
If the position equals `len(tails)`, the number
extends the longest subsequence found so far — append
it. Otherwise, replace `tails[pos]` with the current
number, keeping the tail as small as possible for
length `pos+1`. Return `len(tails)` at the end.

In [ ]:
from typing import List
from collections import defaultdict, deque
import bisect

In [ ]:
def test_harness(func):
    cases = [
        # (nums, expected)
        ([10, 9, 2, 5, 3, 7, 101, 18], 4),
        ([0, 1, 0, 3, 2, 3],           4),
        ([7, 7, 7, 7, 7],              1),
        ([1],                          1),
        ([1, 2, 3, 4, 5],              5),
        ([5, 4, 3, 2, 1],              1),
        ([3, 5, 6, 2, 5, 4, 19, 5, 6, 7, 12], 6),
    ]
    passed = 0
    for nums, expected in cases:
        result = func(nums)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} | nums={nums} |"
                f" got={result} expected={expected}"
            )
    print(f"\nResults: {passed}/{len(cases)} passed")
    if passed == len(cases):
        print("All tests PASSED!")

In [ ]:
def length_of_lis(nums: List[int]) -> int:
    """
    Return length of longest strictly increasing
    subsequence.

    Strategy: patience sort with binary search.
      tails[i] = smallest tail of IS with length i+1.
      For each num:
        pos = bisect_left(tails, num)
        if pos == len(tails): append  (new longest)
        else: tails[pos] = num        (better tail)
      Return len(tails).

    Args:
        nums: list of integers

    Returns:
        int: length of longest increasing subsequence
    """
    print(f"[DEBUG] nums={nums}")

    tails = []

    for num in nums:
        pos = bisect.bisect_left(tails, num)
        print(
            f"[DEBUG] num={num} pos={pos} "
            f"tails={tails}"
        )
        if pos == len(tails):
            tails.append(num)
        else:
            tails[pos] = num

    print(f"[DEBUG] final tails={tails} len={len(tails)}")
    return len(tails)


pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(length_of_lis)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (all subsequences) | O(2^n) | O(n) | Exponential |
| DP (dp[i] = LIS ending at i) | O(n²) | O(n) | Simple, readable |
| Optimal (patience sort) | O(n log n) | O(n) | bisect on sorted tails |

## Real World Connection

At **Citi**, LIS appears in trade sequencing: finding
the longest chain of trades that can settle in strictly
increasing order without re-netting. **AWS** Auto
Scaling uses a conceptually similar pattern to find
the longest monotonic run of metric samples that
justifies scaling up. In **data engineering**, schema
version ordering and migration planning need the
longest chain of compatible upgrades — an LIS variant.
The patience sort algorithm also underlies merge-sort
optimisations in external sort pipelines.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra